# 01 · Basics

rxembed has three verbs: `rx.embed` makes new conformers, `rx.minimize` relaxes coordinates you already have, and
`rx.isomers` lists the species a SMILES or a geometry can be. This notebook uses each on small organic molecules.

In [ ]:
import tempfile
from pathlib import Path

import rxembed as rx
import xyzrender

## Embed a SMILES

`rx.embed` returns an `Ensemble`: one species and its conformers in one RDKit `Mol`. `seed=` makes the run
repeatable.

In [ ]:
ens = rx.embed("OCCCCO", n=8, seed=1)
print(ens)
print(ens.ids)

## Minimize and keep the lowest

`minimize()` relaxes every conformer with UFF and stores its energy in kcal/mol, and `relative()` gives each one
above the lowest. `lowest(3)` returns a new `Ensemble` of the three lowest conformers.

In [ ]:
ens.minimize()
print({cid: round(e, 2) for cid, e in ens.relative().items()})
best = ens.lowest(3)
print(best)

## Write and draw

`dump` writes the conformers to one multi-frame XYZ. `best.mol` is an ordinary RDKit `Mol`, so RDKit's writers work
on it and xyzrender draws its conformers overlaid.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = best.dump(Path(tmp) / "diol.xyz")
    print(Path(path).read_text().splitlines()[:3])

xyzrender.render(xyzrender.load(best.mol, ensemble=True))

## Undefined stereo

`CC=CC(C)O` leaves a double bond and a stereocentre undefined, so it names four species and `rx.embed` refuses it.
`rx.isomers` lists them, and embedding the list gives an `EnsembleSet` with one `Ensemble` per species.

In [ ]:
isomers = rx.isomers("CC=CC(C)O")
print(isomers.summary())

per_species = rx.embed(isomers, n=3, seed=1).minimize()
for species in per_species:
    print(rx.smiles(species.mol))
    display(xyzrender.render(xyzrender.load(species.lowest(1).mol), stereo=True))

## Read an .xyz

`rx.read_xyz` perceives bonds, bond orders and charge with xyzgraph; this file holds a thiourea catalyst beside
glycolamide. `rx.minimize` relaxes the geometry as given, and `rx.changes` confirms that no bond formed or broke
and no stereo changed.

In [ ]:
mol = rx.read_xyz("structures/bimp_small.xyz")
print(rx.smiles(mol))

relaxed = rx.minimize(mol)
print(relaxed)
print(rx.changes(relaxed.mol, relaxed.ids[0]))
xyzrender.render(xyzrender.load(relaxed.mol))

## Next

[02_conformers](02_conformers.ipynb) searches and deduplicates conformers, [03_constraints](03_constraints.ipynb)
holds distances, angles and rigid cores, and [04_nci](04_nci.ipynb) finds binding modes between molecules.